# CODE CNN CƠ BẢN + 3 MÔ HÌNH CẢI TIẾN TRÊN DIABETES RISK

**File dự kiến đặt tại:**

`C:\Users\datng\Desktop\GIÁO TRÌNH\NĂM 4 - KÌ 1\PTCHTTH - Phát Triển Các Hệ Thống Thông Minh\test\a05\data3_diabetes\preprocessor.ipynb`

**Dataset đầu vào:**

`C:\DATA\diabetes_risk_prediction_dataset.csv`

Notebook này triển khai đầy đủ pipeline cho dữ liệu Diabetes:

1. Đọc và khảo sát dữ liệu CSV.
2. Kiểm tra target `Diabetes_Risk`.
3. Kiểm tra missing value, class distribution và các cột có nguy cơ leakage.
4. Loại identifier như `Patient_ID`.
5. Chia train / validation / test **trước khi fit preprocessing**.
6. Xử lý numerical feature:
   - Median imputation.
   - StandardScaler.
7. Xử lý categorical feature:
   - Most-frequent imputation.
   - One-Hot Encoding.
8. Biểu diễn một bệnh nhân:
   - Ban đầu: `N × F`.
   - Sau preprocessing: `N × F_processed`.
   - Trước CNN: `N × 1 × F_processed`.
9. Xây dựng **CNN 1D cơ bản**.
10. Xây dựng **ResNet1D** với `F(x) + x`.
11. Xây dựng **VGGNet1D-style** với nhiều Conv1D kernel 3 xếp chồng.
12. Xây dựng **MobileNet1D-style** với Depthwise Conv1D + Pointwise Conv1D.
13. Huấn luyện 4 mô hình bằng cùng protocol.
14. So sánh Accuracy, Macro F1, Loss, Parameters và Training Time.
15. Confusion Matrix, Classification Report, learning curve.
16. Lưu model và preprocessing pipeline.

> **Lưu ý học thuật:** Diabetes là dữ liệu bảng, không có cấu trúc không gian tự nhiên như ảnh. Việc dùng Conv1D ở đây là một **thử nghiệm kiến trúc** để đáp ứng yêu cầu so sánh CNN; không nên mặc định rằng CNN là lựa chọn tối ưu nhất cho tabular data.

## 1. Import thư viện và kiểm tra GPU

Notebook dùng đúng cơ chế tự chọn GPU như hai notebook trước:

`cuda:0` nếu PyTorch CUDA hoạt động, ngược lại là CPU.

In [ ]:
# Import Path để xử lý đường dẫn file trên Windows.
from pathlib import Path

# Import time để đo thời gian huấn luyện từng mô hình.
import time

# Import copy để lưu lại state_dict tốt nhất theo validation accuracy.
import copy

# Import random để đặt random seed.
import random

# Import pickle để lưu preprocessing pipeline và label encoder.
import pickle

# Import NumPy để xử lý ma trận số.
import numpy as np

# Import Pandas để đọc và khảo sát file CSV.
import pandas as pd

# Import Matplotlib để trực quan hóa phân bố và metric.
import matplotlib.pyplot as plt

# Import PyTorch.
import torch

# Import các layer neural network.
import torch.nn as nn

# Import functional API như ReLU.
import torch.nn.functional as F

# Import TensorDataset để đóng gói tensor feature và label.
from torch.utils.data import TensorDataset

# Import DataLoader để tạo mini-batch.
from torch.utils.data import DataLoader

# Import train_test_split để chia train/validation/test.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer để áp dụng preprocessing khác nhau cho numerical/categorical.
from sklearn.compose import ColumnTransformer

# Import Pipeline để ghép imputer -> scaler / encoder.
from sklearn.pipeline import Pipeline

# Import SimpleImputer để xử lý missing value.
from sklearn.impute import SimpleImputer

# Import StandardScaler để chuẩn hóa numerical feature.
from sklearn.preprocessing import StandardScaler

# Import OneHotEncoder để mã hóa categorical feature.
from sklearn.preprocessing import OneHotEncoder

# Import LabelEncoder để biến target dạng chữ thành class index.
from sklearn.preprocessing import LabelEncoder

# Import các metric đánh giá.
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Seed cố định để tăng khả năng tái lập.
SEED = 42

# Đặt seed cho Python.
random.seed(SEED)

# Đặt seed cho NumPy.
np.random.seed(SEED)

# Đặt seed cho PyTorch CPU.
torch.manual_seed(SEED)

# Nếu có CUDA thì đặt seed cho GPU.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Tăng khả năng deterministic.
torch.backends.cudnn.deterministic = True

# Tắt benchmark để tránh thay đổi thuật toán giữa các lần chạy.
torch.backends.cudnn.benchmark = False

# Chọn GPU NVIDIA nếu CUDA khả dụng.
DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

# In thông tin môi trường để chắc chắn model đang chạy ở đâu.
print("PyTorch version :", torch.__version__)
print("CUDA build      :", torch.version.cuda)
print("CUDA available  :", torch.cuda.is_available())
print("DEVICE          :", DEVICE)

# Nếu CUDA hoạt động thì in tên GPU.
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
else:
    print(
        "CẢNH BÁO: Notebook đang dùng CPU. "
        "Nếu máy có RTX 3060, hãy kiểm tra lại PyTorch CUDA build."
    )

## 2. Khai báo đường dẫn dataset

In [ ]:
# Đường dẫn tuyệt đối tới file CSV Diabetes.
DATA_FILE = Path(
    r"C:\DATA\diabetes_risk_prediction_dataset.csv"
)

# Kiểm tra file có tồn tại hay không.
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy dataset tại:\n{DATA_FILE}"
    )

# In đường dẫn xác nhận.
print("DATA_FILE =", DATA_FILE)

## 3. Đọc dataset và khảo sát cấu trúc ban đầu

In [ ]:
# Đọc toàn bộ dataset bằng Pandas.
df = pd.read_csv(DATA_FILE)

# In số hàng và số cột.
print("df.shape =", df.shape)

# Hiển thị 5 dòng đầu.
display(df.head())

# In tên tất cả cột.
print("\nDanh sách cột:")
for index, column in enumerate(df.columns):
    print(f"{index:02d}. {column}")

# In kiểu dữ liệu từng cột.
print("\nKiểu dữ liệu:")
display(
    df.dtypes
    .rename("dtype")
    .to_frame()
)

## 4. Xác định target `Diabetes_Risk`

Theo tài liệu bài tập, target chính là `Diabetes_Risk`.

Cell này cố định target để tránh vô tình train vào một cột khác.

In [ ]:
# Tên target theo mô tả dataset trong tài liệu.
TARGET_COLUMN = "Diabetes_Risk"

# Kiểm tra target có thực sự tồn tại.
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Không tìm thấy target '{TARGET_COLUMN}'.\n"
        f"Các cột hiện có: {df.columns.tolist()}"
    )

# In các giá trị target duy nhất.
print(
    "Target classes:",
    df[TARGET_COLUMN]
    .dropna()
    .unique()
)

# In số lượng class.
print(
    "Số class:",
    df[TARGET_COLUMN]
    .dropna()
    .nunique()
)

## 5. Kiểm tra missing value và duplicate

Đây là bước khảo sát dữ liệu trước preprocessing.

In [ ]:
# Đếm missing value theo từng cột.
missing_count = (
    df.isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

# Chỉ hiển thị cột có missing.
missing_nonzero = missing_count[
    missing_count > 0
]

# In số cột có missing.
print(
    "Số cột có missing:",
    len(missing_nonzero)
)

# Hiển thị bảng missing.
display(
    missing_nonzero
    .rename("missing_count")
    .to_frame()
)

# Đếm số dòng duplicate hoàn toàn.
duplicate_count = int(
    df.duplicated().sum()
)

# In duplicate count.
print(
    "Số dòng duplicate hoàn toàn:",
    duplicate_count
)

## 6. Phân bố target

Với dữ liệu y tế/risk classification, cần kiểm tra class imbalance trước khi chỉ nhìn Accuracy.

In [ ]:
# Đếm số mẫu theo target.
target_counts = (
    df[TARGET_COLUMN]
    .value_counts(
        dropna=False
    )
)

# Hiển thị bảng.
display(
    target_counts
    .rename("count")
    .to_frame()
)

# Vẽ biểu đồ phân bố target.
plt.figure(
    figsize=(8, 4)
)

# Bar chart.
plt.bar(
    target_counts.index.astype(str),
    target_counts.values,
)

# Label trục.
plt.xlabel("Diabetes_Risk")
plt.ylabel("Số bệnh nhân")

# Title.
plt.title(
    "Phân bố nhãn Diabetes_Risk"
)

# Grid ngang.
plt.grid(
    axis="y",
    alpha=0.25,
)

# Hiển thị.
plt.show()

## 7. Kiểm tra identifier và nguy cơ data leakage

Theo tài liệu bài tập:

- Các cột định danh như `Patient_ID` không nên dùng làm feature.
- Cần đặc biệt kiểm tra các cột được tạo trực tiếp/gián tiếp từ target.

Notebook **tự động loại identifier phổ biến** và tự động cảnh báo các tên cột có từ khóa `risk`, `score`, `target`, `label`, `outcome`.

Ngoài ra, nếu tồn tại cột **`Diabetes_Risk_Score`**, notebook mặc định loại cột này để tránh khả năng leakage quá trực tiếp.

In [ ]:
# Các tên cột identifier thường gặp.
IDENTIFIER_CANDIDATES = {
    "patient_id",
    "patientid",
    "id",
    "record_id",
    "recordid",
}

# Hàm chuẩn hóa tên cột để so sánh identifier.
def normalize_column_name(column_name: str) -> str:
    # Chuyển về chữ thường.
    normalized = column_name.strip().lower()

    # Bỏ space.
    normalized = normalized.replace(" ", "_")

    # Bỏ dấu gạch ngang.
    normalized = normalized.replace("-", "_")

    # Trả kết quả.
    return normalized

# Tìm identifier column.
identifier_columns = [
    column
    for column in df.columns
    if (
        normalize_column_name(column)
        in IDENTIFIER_CANDIDATES
        and column != TARGET_COLUMN
    )
]

# Các từ khóa dễ liên quan tới leakage.
LEAKAGE_KEYWORDS = [
    "risk",
    "score",
    "target",
    "label",
    "outcome",
]

# Tìm cột đáng nghi ngoài target.
suspicious_columns = [
    column
    for column in df.columns
    if (
        column != TARGET_COLUMN
        and any(
            keyword in column.lower()
            for keyword in LEAKAGE_KEYWORDS
        )
    )
]

# Danh sách leakage mặc định sẽ loại nếu tồn tại.
DEFAULT_LEAKAGE_COLUMNS = [
    "Diabetes_Risk_Score",
]

# Chỉ giữ những cột thực sự tồn tại.
leakage_columns_to_drop = [
    column
    for column in DEFAULT_LEAKAGE_COLUMNS
    if column in df.columns
]

# In identifier.
print(
    "Identifier columns sẽ loại:",
    identifier_columns
)

# In suspicious columns để người học tự kiểm tra.
print(
    "\nCác cột cần REVIEW về data leakage:",
    suspicious_columns
)

# In leakage columns mặc định drop.
print(
    "\nCác cột leakage mặc định sẽ loại:",
    leakage_columns_to_drop
)

### Tùy chọn chỉnh danh sách leakage

Nếu sau khi xem ý nghĩa dataset bạn xác định thêm một cột được tính trực tiếp từ `Diabetes_Risk`, hãy thêm tên cột vào `EXTRA_COLUMNS_TO_DROP`.

In [ ]:
# Có thể thêm thủ công các cột muốn loại.
EXTRA_COLUMNS_TO_DROP = [
    # Ví dụ:
    # "Some_Target_Derived_Column",
]

# Tạo danh sách cột cuối cùng cần loại.
columns_to_drop = list(
    dict.fromkeys(
        identifier_columns
        + leakage_columns_to_drop
        + EXTRA_COLUMNS_TO_DROP
    )
)

# In danh sách cuối.
print(
    "Columns to drop:",
    columns_to_drop
)

## 8. Tạo X và y

- `X`: toàn bộ feature sau khi bỏ target, ID và leakage columns.
- `y`: `Diabetes_Risk`.

In [ ]:
# Bỏ các dòng target bị missing vì không thể train supervised nếu không có nhãn.
model_df = df.dropna(
    subset=[TARGET_COLUMN]
).copy()

# Tạo y dạng chuỗi để LabelEncoder xử lý ổn định.
y_raw = (
    model_df[TARGET_COLUMN]
    .astype(str)
)

# Tạo X bằng cách bỏ target và các cột không dùng.
X_raw = model_df.drop(
    columns=[
        TARGET_COLUMN,
        *columns_to_drop,
    ],
    errors="ignore",
)

# In shape feature.
print(
    "X_raw.shape =",
    X_raw.shape,
)

# In shape label.
print(
    "y_raw.shape =",
    y_raw.shape,
)

# In số feature thô.
print(
    "Số feature thô =",
    X_raw.shape[1],
)

## 9. Chia Train / Validation / Test **trước khi fit preprocessing**

Tỷ lệ:

- 80% train
- 10% validation
- 10% test

Việc fit imputer/scaler/encoder sau khi split giúp tránh data leakage từ validation/test.

In [ ]:
# Tạo toàn bộ row index.
all_indices = np.arange(
    len(model_df)
)

# Chia 80% train và 20% temporary.
train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_raw.to_numpy(),
)

# Lấy target phần temporary.
temp_targets = y_raw.iloc[
    temp_indices
].to_numpy()

# Chia temporary thành validation và test bằng nhau.
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_targets,
)

# Tạo DataFrame train.
X_train_raw = X_raw.iloc[
    train_indices
].copy()

# Tạo target train.
y_train_raw = y_raw.iloc[
    train_indices
].copy()

# Validation.
X_val_raw = X_raw.iloc[
    val_indices
].copy()

# Validation target.
y_val_raw = y_raw.iloc[
    val_indices
].copy()

# Test.
X_test_raw = X_raw.iloc[
    test_indices
].copy()

# Test target.
y_test_raw = y_raw.iloc[
    test_indices
].copy()

# In số mẫu.
print(
    "Train:",
    len(X_train_raw)
)
print(
    "Validation:",
    len(X_val_raw)
)
print(
    "Test:",
    len(X_test_raw)
)

## 10. Encode target

`CrossEntropyLoss` yêu cầu target là class index kiểu integer:

`0, 1, 2, ...`

In [ ]:
# Khởi tạo LabelEncoder.
label_encoder = LabelEncoder()

# Fit CHỈ trên target train.
y_train = label_encoder.fit_transform(
    y_train_raw
)

# Transform validation bằng mapping đã học từ train.
y_val = label_encoder.transform(
    y_val_raw
)

# Transform test.
y_test = label_encoder.transform(
    y_test_raw
)

# Lấy tên class theo thứ tự index.
CLASS_NAMES = label_encoder.classes_.tolist()

# Số class.
NUM_CLASSES = len(
    CLASS_NAMES
)

# In mapping.
print(
    "Class mapping:"
)

# Duyệt từng class.
for class_index, class_name in enumerate(
    CLASS_NAMES
):
    print(
        f"{class_index}: {class_name}"
    )

# Kiểm tra có ít nhất 2 lớp.
if NUM_CLASSES < 2:
    raise ValueError(
        "Target chỉ có một lớp; không thể huấn luyện classification."
    )

## 11. Xác định numerical và categorical features

In [ ]:
# Numerical columns gồm kiểu number.
numerical_columns = (
    X_train_raw
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

# Categorical là tất cả cột còn lại.
categorical_columns = [
    column
    for column in X_train_raw.columns
    if column not in numerical_columns
]

# In số numerical.
print(
    "Numerical columns:",
    len(numerical_columns)
)

# In danh sách numerical.
print(
    numerical_columns
)

# In số categorical.
print(
    "\nCategorical columns:",
    len(categorical_columns)
)

# In danh sách categorical.
print(
    categorical_columns
)

## 12. Tạo preprocessing pipeline

### Numerical
`Median Imputer → StandardScaler`

### Categorical
`Most-frequent Imputer → OneHotEncoder`

Notebook có helper tương thích cả scikit-learn mới (`sparse_output=False`) và bản cũ (`sparse=False`).

In [ ]:
# Helper tạo OneHotEncoder dense tương thích nhiều phiên bản sklearn.
def make_dense_one_hot_encoder():
    # Thử API mới.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )

    # Nếu sklearn cũ không có sparse_output.
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# Pipeline numerical.
numeric_pipeline = Pipeline(
    steps=[
        # Missing numerical -> median của train.
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),

        # Chuẩn hóa numerical về mean~0, std~1.
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

# Pipeline categorical.
categorical_pipeline = Pipeline(
    steps=[
        # Missing categorical -> mode.
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),

        # One-hot encode.
        (
            "onehot",
            make_dense_one_hot_encoder(),
        ),
    ]
)

# Danh sách transformer thực tế.
transformers_list = []

# Chỉ thêm numerical transformer nếu có numerical columns.
if numerical_columns:
    transformers_list.append(
        (
            "num",
            numeric_pipeline,
            numerical_columns,
        )
    )

# Chỉ thêm categorical transformer nếu có categorical columns.
if categorical_columns:
    transformers_list.append(
        (
            "cat",
            categorical_pipeline,
            categorical_columns,
        )
    )

# ColumnTransformer ghép hai loại feature.
preprocessor = ColumnTransformer(
    transformers=transformers_list,

    # Bỏ mọi cột không được khai báo.
    remainder="drop",

    # Ép output thành dense khi có thể.
    sparse_threshold=0.0,
)

# Fit preprocessing chỉ trên train.
X_train_processed = preprocessor.fit_transform(
    X_train_raw
)

# Transform validation.
X_val_processed = preprocessor.transform(
    X_val_raw
)

# Transform test.
X_test_processed = preprocessor.transform(
    X_test_raw
)

# Ép về NumPy float32 cho PyTorch.
X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32,
)

# Validation float32.
X_val_processed = np.asarray(
    X_val_processed,
    dtype=np.float32,
)

# Test float32.
X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32,
)

# In shape sau preprocessing.
print(
    "X_train_processed:",
    X_train_processed.shape,
)
print(
    "X_val_processed  :",
    X_val_processed.shape,
)
print(
    "X_test_processed :",
    X_test_processed.shape,
)

## 13. Kiểm tra feature names sau preprocessing

Bước này giúp biết `F_processed` thực tế là bao nhiêu sau One-Hot Encoding.

In [ ]:
# Thử lấy tên feature output từ ColumnTransformer.
try:
    # API sklearn hiện đại.
    processed_feature_names = (
        preprocessor
        .get_feature_names_out()
        .tolist()
    )

# Nếu version cũ không hỗ trợ thì tạo tên generic.
except Exception:
    processed_feature_names = [
        f"feature_{index}"
        for index in range(
            X_train_processed.shape[1]
        )
    ]

# Số feature sau preprocessing.
NUM_FEATURES = (
    X_train_processed.shape[1]
)

# In số feature.
print(
    "NUM_FEATURES =",
    NUM_FEATURES,
)

# Hiển thị tối đa 50 feature name đầu.
print(
    "\nMột số feature sau preprocessing:"
)

for feature_name in processed_feature_names[:50]:
    print(
        "-",
        feature_name,
    )

## 14. Biểu diễn tabular data cho Conv1D

Sau preprocessing:

`N × F_processed`

Conv1D yêu cầu:

`N × C × L`

Ta coi:

- `C = 1`
- `L = F_processed`

nên reshape:

`N × F_processed → N × 1 × F_processed`

In [ ]:
# Thêm channel dimension cho train.
X_train_cnn = np.expand_dims(
    X_train_processed,
    axis=1,
)

# Thêm channel dimension cho validation.
X_val_cnn = np.expand_dims(
    X_val_processed,
    axis=1,
)

# Thêm channel dimension cho test.
X_test_cnn = np.expand_dims(
    X_test_processed,
    axis=1,
)

# In shape.
print(
    "Train CNN shape:",
    X_train_cnn.shape,
)
print(
    "Val CNN shape  :",
    X_val_cnn.shape,
)
print(
    "Test CNN shape :",
    X_test_cnn.shape,
)

# Kiểm tra channel dimension bằng 1.
assert (
    X_train_cnn.shape[1]
    == 1
)

## 15. Chuyển sang TensorDataset và DataLoader

In [ ]:
# Convert train features sang tensor float32.
train_x_tensor = torch.from_numpy(
    X_train_cnn
).float()

# Convert train target sang LongTensor.
train_y_tensor = torch.from_numpy(
    y_train.astype(
        np.int64
    )
).long()

# Validation feature.
val_x_tensor = torch.from_numpy(
    X_val_cnn
).float()

# Validation label.
val_y_tensor = torch.from_numpy(
    y_val.astype(
        np.int64
    )
).long()

# Test feature.
test_x_tensor = torch.from_numpy(
    X_test_cnn
).float()

# Test label.
test_y_tensor = torch.from_numpy(
    y_test.astype(
        np.int64
    )
).long()

# Tạo TensorDataset train.
train_dataset = TensorDataset(
    train_x_tensor,
    train_y_tensor,
)

# Validation dataset.
val_dataset = TensorDataset(
    val_x_tensor,
    val_y_tensor,
)

# Test dataset.
test_dataset = TensorDataset(
    test_x_tensor,
    test_y_tensor,
)

# Batch size dùng chung cho 4 model.
BATCH_SIZE = 256

# Windows/Jupyter ổn định với num_workers=0.
NUM_WORKERS = 0

# Pin memory khi chạy CUDA.
PIN_MEMORY = (
    DEVICE.type == "cuda"
)

# Train loader.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# Validation loader.
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# Test loader.
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# Kiểm tra một mini-batch.
sample_features, sample_labels = next(
    iter(train_loader)
)

# In shape [batch,1,F].
print(
    "sample_features.shape =",
    sample_features.shape,
)

# In shape label [batch].
print(
    "sample_labels.shape   =",
    sample_labels.shape,
)

# PHẦN B — CNN CƠ BẢN + 3 MÔ HÌNH CẢI TIẾN

**Quan trọng:** đây là các phiên bản **1D analog** của tư tưởng CNN/ResNet/VGG/MobileNet để áp dụng trên vector feature tabular.

## 16. Mô hình 1 — CNN 1D cơ bản

Pipeline:

`Conv1D → ReLU → Conv1D → ReLU → Adaptive Pool → Linear`

In [ ]:
# CNN 1D baseline.
class BasicCNN1D(nn.Module):
    # Constructor.
    def __init__(
        self,
        num_classes: int,
    ):
        # Khởi tạo nn.Module.
        super().__init__()

        # Conv1D đầu tiên: 1 channel -> 32 channel.
        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1,
        )

        # Conv1D thứ hai: 32 -> 64 channel.
        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1,
        )

        # AdaptiveAvgPool1d đưa chiều feature sequence về 1.
        self.pool = nn.AdaptiveAvgPool1d(
            output_size=1
        )

        # Dropout chống overfitting.
        self.dropout = nn.Dropout(
            p=0.30
        )

        # Linear cuối tạo logits.
        self.fc = nn.Linear(
            64,
            num_classes,
        )

    # Forward.
    def forward(self, x):
        # Conv1D 1.
        x = self.conv1(x)

        # ReLU.
        x = F.relu(x)

        # Conv1D 2.
        x = self.conv2(x)

        # ReLU.
        x = F.relu(x)

        # Global average pool -> [N,64,1].
        x = self.pool(x)

        # Flatten -> [N,64].
        x = torch.flatten(
            x,
            start_dim=1,
        )

        # Dropout.
        x = self.dropout(x)

        # Logits.
        logits = self.fc(x)

        # Return.
        return logits

## 17. Mô hình 2 — ResNet1D

Công thức vẫn là:

`y = F(x) + shortcut(x)`

Projection shortcut dùng Conv1D `1×1` khi channel hoặc length thay đổi.

In [ ]:
# Basic residual block 1D.
class BasicBlock1D(nn.Module):
    # Constructor.
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
    ):
        # Khởi tạo.
        super().__init__()

        # Conv1D đầu.
        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        # BatchNorm 1D.
        self.bn1 = nn.BatchNorm1d(
            out_channels
        )

        # Conv1D thứ hai.
        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        # BatchNorm thứ hai.
        self.bn2 = nn.BatchNorm1d(
            out_channels
        )

        # Nếu shape thay đổi thì projection shortcut.
        if (
            stride != 1
            or in_channels != out_channels
        ):
            self.shortcut = nn.Sequential(
                # Conv1D kernel 1 để khớp channel/length.
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),

                # Normalize shortcut.
                nn.BatchNorm1d(
                    out_channels
                ),
            )
        else:
            # Shape giống nhau -> identity.
            self.shortcut = nn.Identity()

    # Forward.
    def forward(self, x):
        # Shortcut.
        identity = self.shortcut(x)

        # Residual conv1.
        out = self.conv1(x)

        # BN.
        out = self.bn1(out)

        # ReLU.
        out = F.relu(out)

        # Residual conv2.
        out = self.conv2(out)

        # BN.
        out = self.bn2(out)

        # F(x) + shortcut(x).
        out = out + identity

        # ReLU sau cộng.
        out = F.relu(out)

        # Return.
        return out

In [ ]:
# ResNet1D cho Diabetes.
class ResNet1D(nn.Module):
    # Constructor.
    def __init__(
        self,
        num_classes: int,
    ):
        # Khởi tạo.
        super().__init__()

        # Stem 1 -> 32 channel.
        self.stem = nn.Sequential(
            nn.Conv1d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )

        # Stage 1 giữ length.
        self.layer1 = nn.Sequential(
            BasicBlock1D(
                32,
                32,
            ),
            BasicBlock1D(
                32,
                32,
            ),
        )

        # Stage 2 giảm length khoảng một nửa.
        self.layer2 = nn.Sequential(
            BasicBlock1D(
                32,
                64,
                stride=2,
            ),
            BasicBlock1D(
                64,
                64,
            ),
        )

        # Stage 3 giảm tiếp.
        self.layer3 = nn.Sequential(
            BasicBlock1D(
                64,
                128,
                stride=2,
            ),
            BasicBlock1D(
                128,
                128,
            ),
        )

        # Global average pooling.
        self.pool = nn.AdaptiveAvgPool1d(
            1
        )

        # Linear classifier.
        self.fc = nn.Linear(
            128,
            num_classes,
        )

    # Forward.
    def forward(self, x):
        # Stem.
        x = self.stem(x)

        # Residual stage 1.
        x = self.layer1(x)

        # Residual stage 2.
        x = self.layer2(x)

        # Residual stage 3.
        x = self.layer3(x)

        # Global pool.
        x = self.pool(x)

        # Flatten.
        x = torch.flatten(
            x,
            start_dim=1,
        )

        # Logits.
        logits = self.fc(x)

        # Return.
        return logits

## 18. Mô hình 3 — VGGNet1D-style

Tư tưởng VGG được chuyển sang 1D:

`nhiều Conv1D kernel=3 xếp chồng → pooling → tăng channel`

In [ ]:
# Tạo VGG block 1D.
def make_vgg1d_block(
    in_channels: int,
    out_channels: int,
    num_convs: int,
):
    # List layer.
    layers = []

    # Thêm nhiều Conv1D kernel 3.
    for conv_index in range(
        num_convs
    ):
        # Input channel của conv hiện tại.
        current_in = (
            in_channels
            if conv_index == 0
            else out_channels
        )

        # Conv1D kernel 3.
        layers.append(
            nn.Conv1d(
                current_in,
                out_channels,
                kernel_size=3,
                padding=1,
            )
        )

        # ReLU.
        layers.append(
            nn.ReLU(inplace=True)
        )

    # Pool giảm sequence length.
    layers.append(
        nn.MaxPool1d(
            kernel_size=2,
            stride=2,
            ceil_mode=True,
        )
    )

    # Return Sequential.
    return nn.Sequential(
        *layers
    )

In [ ]:
# VGG-style 1D.
class VGGNet1D(nn.Module):
    # Constructor.
    def __init__(
        self,
        num_classes: int,
    ):
        # Khởi tạo.
        super().__init__()

        # Block 1.
        self.block1 = make_vgg1d_block(
            1,
            32,
            2,
        )

        # Block 2.
        self.block2 = make_vgg1d_block(
            32,
            64,
            2,
        )

        # Block 3.
        self.block3 = make_vgg1d_block(
            64,
            128,
            3,
        )

        # Global average pool.
        self.pool = nn.AdaptiveAvgPool1d(
            1
        )

        # Classifier.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                128,
                128,
            ),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(
                128,
                num_classes,
            ),
        )

    # Forward.
    def forward(self, x):
        # VGG block 1.
        x = self.block1(x)

        # VGG block 2.
        x = self.block2(x)

        # VGG block 3.
        x = self.block3(x)

        # Global pool.
        x = self.pool(x)

        # Logits.
        logits = self.classifier(x)

        # Return.
        return logits

## 19. Mô hình 4 — MobileNet1D-style

Tương tự MobileNet v1:

- **Depthwise Conv1D**: `groups=in_channels`
- **Pointwise Conv1D**: `kernel_size=1`

In [ ]:
# Depthwise separable block 1D.
class DepthwiseSeparableBlock1D(
    nn.Module
):
    # Constructor.
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
    ):
        # Khởi tạo.
        super().__init__()

        # Depthwise Conv1D.
        self.depthwise = nn.Sequential(
            nn.Conv1d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False,
            ),
            nn.BatchNorm1d(
                in_channels
            ),
            nn.ReLU(inplace=True),
        )

        # Pointwise Conv1D kernel=1.
        self.pointwise = nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm1d(
                out_channels
            ),
            nn.ReLU(inplace=True),
        )

    # Forward.
    def forward(self, x):
        # Spatial/sequence filtering từng channel.
        x = self.depthwise(x)

        # Channel mixing.
        x = self.pointwise(x)

        # Return.
        return x

In [ ]:
# MobileNet1D-style.
class MobileNet1D(nn.Module):
    # Constructor.
    def __init__(
        self,
        num_classes: int,
    ):
        # Khởi tạo.
        super().__init__()

        # Standard Conv1D đầu.
        self.stem = nn.Sequential(
            nn.Conv1d(
                1,
                32,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
        )

        # Block 1 giữ length.
        self.block1 = DepthwiseSeparableBlock1D(
            32,
            64,
            stride=1,
        )

        # Block 2 giảm length.
        self.block2 = DepthwiseSeparableBlock1D(
            64,
            128,
            stride=2,
        )

        # Block 3 giữ.
        self.block3 = DepthwiseSeparableBlock1D(
            128,
            128,
            stride=1,
        )

        # Block 4 giảm tiếp.
        self.block4 = DepthwiseSeparableBlock1D(
            128,
            256,
            stride=2,
        )

        # Global average pool.
        self.pool = nn.AdaptiveAvgPool1d(
            1
        )

        # Classifier.
        self.fc = nn.Linear(
            256,
            num_classes,
        )

    # Forward.
    def forward(self, x):
        # Stem.
        x = self.stem(x)

        # Block 1.
        x = self.block1(x)

        # Block 2.
        x = self.block2(x)

        # Block 3.
        x = self.block3(x)

        # Block 4.
        x = self.block4(x)

        # Global pool.
        x = self.pool(x)

        # Flatten.
        x = torch.flatten(
            x,
            start_dim=1,
        )

        # Logits.
        logits = self.fc(x)

        # Return.
        return logits

## 20. Kiểm tra forward pass và số tham số

In [ ]:
# Hàm đếm learnable parameters.
def count_parameters(
    model: nn.Module,
) -> int:
    # Cộng số phần tử của tất cả parameter trainable.
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

# Khởi tạo 4 model.
models = {
    "CNN cơ bản": BasicCNN1D(
        NUM_CLASSES
    ),
    "ResNet": ResNet1D(
        NUM_CLASSES
    ),
    "VGGNet": VGGNet1D(
        NUM_CLASSES
    ),
    "MobileNet": MobileNet1D(
        NUM_CLASSES
    ),
}

# Dummy input có đúng số feature sau preprocessing.
dummy_x = torch.randn(
    4,
    1,
    NUM_FEATURES,
)

# Kiểm tra từng model.
for model_name, model in models.items():
    # Không cần gradient.
    with torch.no_grad():
        # Forward.
        dummy_logits = model(
            dummy_x
        )

    # In tên.
    print(
        f"\n{model_name}"
    )

    # In shape output.
    print(
        "Output shape:",
        tuple(
            dummy_logits.shape
        ),
    )

    # In số parameter.
    print(
        "Parameters:",
        f"{count_parameters(model):,}",
    )

    # Output phải [batch, NUM_CLASSES].
    assert (
        dummy_logits.shape
        == (
            4,
            NUM_CLASSES,
        )
    )

# PHẦN C — TRAINING, ĐÁNH GIÁ, SO SÁNH, VISUALIZATION

## 21. Hàm train một epoch

In [ ]:
# Train một epoch.
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
):
    # Training mode.
    model.train()

    # Tổng loss.
    running_loss = 0.0

    # Tổng đúng.
    running_correct = 0

    # Tổng mẫu.
    running_total = 0

    # Duyệt mini-batch.
    for features, labels in loader:
        # Đưa feature lên device.
        features = features.to(
            device,
            non_blocking=True,
        )

        # Đưa label lên device.
        labels = labels.to(
            device,
            non_blocking=True,
        )

        # Xóa gradient cũ.
        optimizer.zero_grad(
            set_to_none=True
        )

        # Forward.
        logits = model(
            features
        )

        # Loss.
        loss = criterion(
            logits,
            labels,
        )

        # Backprop.
        loss.backward()

        # Update weight.
        optimizer.step()

        # Batch size.
        batch_size = features.size(
            0
        )

        # Cộng loss.
        running_loss += (
            loss.item()
            * batch_size
        )

        # Prediction.
        predictions = logits.argmax(
            dim=1
        )

        # Đếm đúng.
        running_correct += (
            predictions == labels
        ).sum().item()

        # Cộng số mẫu.
        running_total += batch_size

    # Average loss.
    epoch_loss = (
        running_loss
        / running_total
    )

    # Accuracy.
    epoch_acc = (
        running_correct
        / running_total
    )

    # Return.
    return (
        epoch_loss,
        epoch_acc,
    )

## 22. Hàm evaluate

In [ ]:
# Không lưu gradient khi evaluate.
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
):
    # Eval mode.
    model.eval()

    # Tổng loss.
    running_loss = 0.0

    # Tổng đúng.
    running_correct = 0

    # Tổng mẫu.
    running_total = 0

    # Duyệt batch.
    for features, labels in loader:
        # Feature lên device.
        features = features.to(
            device,
            non_blocking=True,
        )

        # Label lên device.
        labels = labels.to(
            device,
            non_blocking=True,
        )

        # Forward.
        logits = model(
            features
        )

        # Loss.
        loss = criterion(
            logits,
            labels,
        )

        # Prediction.
        predictions = logits.argmax(
            dim=1
        )

        # Batch size.
        batch_size = features.size(
            0
        )

        # Cộng loss.
        running_loss += (
            loss.item()
            * batch_size
        )

        # Cộng đúng.
        running_correct += (
            predictions == labels
        ).sum().item()

        # Cộng tổng mẫu.
        running_total += batch_size

    # Loss trung bình.
    avg_loss = (
        running_loss
        / running_total
    )

    # Accuracy.
    avg_acc = (
        running_correct
        / running_total
    )

    # Return.
    return (
        avg_loss,
        avg_acc,
    )

## 23. Hàm fit nhiều epoch

In [ ]:
# Train hoàn chỉnh một model.
def fit_model(
    model_name: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int,
    learning_rate: float,
):
    # Đưa model lên GPU/CPU.
    model = model.to(
        device
    )

    # CrossEntropyLoss cho multiclass/binary-as-2-class.
    criterion = nn.CrossEntropyLoss()

    # Adam optimizer.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    # Lưu lịch sử.
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    # Best validation accuracy.
    best_val_acc = -1.0

    # Best state.
    best_state = None

    # Bắt đầu đo thời gian.
    start_time = time.perf_counter()

    # Lặp epoch.
    for epoch in range(
        1,
        epochs + 1,
    ):
        # Train.
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )

        # Validate.
        val_loss, val_acc = evaluate(
            model,
            val_loader,
            criterion,
            device,
        )

        # Lưu metric.
        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_acc"
        ].append(
            val_acc
        )

        # Nếu val accuracy tốt hơn.
        if val_acc > best_val_acc:
            # Cập nhật best.
            best_val_acc = val_acc

            # Copy state.
            best_state = copy.deepcopy(
                model.state_dict()
            )

        # In tiến trình.
        print(
            f"[{model_name}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f}"
        )

    # Tổng thời gian.
    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    # Load best state.
    model.load_state_dict(
        best_state
    )

    # Return.
    return (
        model,
        history,
        elapsed_seconds,
        best_val_acc,
    )

## 24. Hyperparameter

Dataset có khoảng 50k mẫu nên mặc định dùng 15 epoch.  
Nếu chỉ kiểm tra pipeline, bật `FAST_DEBUG=True`.

In [ ]:
# Số epoch chính thức.
EPOCHS = 15

# Learning rate.
LEARNING_RATE = 1e-3

# Debug mode.
FAST_DEBUG = False

# Nếu debug thì chỉ chạy 1 epoch.
if FAST_DEBUG:
    EPOCHS = 1

# In config.
print(
    "EPOCHS        =",
    EPOCHS,
)
print(
    "LEARNING_RATE =",
    LEARNING_RATE,
)
print(
    "BATCH_SIZE    =",
    BATCH_SIZE,
)
print(
    "DEVICE        =",
    DEVICE,
)

## 25. Huấn luyện cả 4 mô hình

In [ ]:
# Model đã train.
trained_models = {}

# Learning histories.
histories = {}

# Kết quả tổng hợp.
results = {}

# Duyệt 4 model.
for model_name, model in models.items():
    # Separator.
    print(
        "\n"
        + "=" * 80
    )

    # Tên model.
    print(
        "TRAINING:",
        model_name,
    )

    # Parameter count.
    print(
        "Parameters:",
        f"{count_parameters(model):,}",
    )

    # Train.
    (
        trained_model,
        history,
        train_seconds,
        best_val_acc,
    ) = fit_model(
        model_name=model_name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
    )

    # Loss function test.
    criterion = nn.CrossEntropyLoss()

    # Test loss/accuracy.
    test_loss, test_acc = evaluate(
        trained_model,
        test_loader,
        criterion,
        DEVICE,
    )

    # Lưu model.
    trained_models[
        model_name
    ] = trained_model

    # Lưu history.
    histories[
        model_name
    ] = history

    # Lưu metric cơ bản.
    results[
        model_name
    ] = {
        "parameters": count_parameters(
            trained_model
        ),
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "train_seconds": train_seconds,
    }

    # In test result.
    print(
        f"TEST | "
        f"loss={test_loss:.4f} | "
        f"accuracy={test_acc:.4f} | "
        f"time={train_seconds:.1f}s"
    )

## 26. Hàm lấy toàn bộ prediction

In [ ]:
# Không gradient.
@torch.no_grad()
def predict_all(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    # Eval mode.
    model.eval()

    # Target list.
    all_targets = []

    # Prediction list.
    all_predictions = []

    # Duyệt batch.
    for features, labels in loader:
        # Feature lên device.
        features = features.to(
            device,
            non_blocking=True,
        )

        # Forward.
        logits = model(
            features
        )

        # Prediction.
        predictions = logits.argmax(
            dim=1
        )

        # Lưu target.
        all_targets.append(
            labels.numpy()
        )

        # Lưu prediction về CPU.
        all_predictions.append(
            predictions.cpu().numpy()
        )

    # Ghép target.
    y_true = np.concatenate(
        all_targets
    )

    # Ghép prediction.
    y_pred = np.concatenate(
        all_predictions
    )

    # Return.
    return (
        y_true,
        y_pred,
    )

## 27. Tính Macro F1 và tạo bảng so sánh

Macro F1 hữu ích khi các lớp không cân bằng vì mỗi lớp được tính trọng số ngang nhau.

In [ ]:
# List record.
comparison_records = []

# Duyệt từng model.
for model_name, model in trained_models.items():
    # Prediction test.
    y_true, y_pred = predict_all(
        model,
        test_loader,
        DEVICE,
    )

    # Macro F1.
    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    # Lưu thêm vào result.
    results[
        model_name
    ][
        "macro_f1"
    ] = macro_f1

    # Tạo row.
    comparison_records.append(
        {
            "Model": model_name,
            "Parameters": results[
                model_name
            ][
                "parameters"
            ],
            "Best Val Accuracy (%)": (
                results[
                    model_name
                ][
                    "best_val_acc"
                ]
                * 100.0
            ),
            "Test Accuracy (%)": (
                results[
                    model_name
                ][
                    "test_acc"
                ]
                * 100.0
            ),
            "Macro F1 (%)": (
                macro_f1
                * 100.0
            ),
            "Test Loss": results[
                model_name
            ][
                "test_loss"
            ],
            "Train Time (s)": results[
                model_name
            ][
                "train_seconds"
            ],
        }
    )

# DataFrame.
comparison_df = pd.DataFrame(
    comparison_records
)

# Sort theo Test Accuracy.
comparison_df = (
    comparison_df
    .sort_values(
        by="Test Accuracy (%)",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Hiển thị.
display(
    comparison_df.style.format(
        {
            "Parameters": "{:,.0f}",
            "Best Val Accuracy (%)": "{:.2f}",
            "Test Accuracy (%)": "{:.2f}",
            "Macro F1 (%)": "{:.2f}",
            "Test Loss": "{:.4f}",
            "Train Time (s)": "{:.2f}",
        }
    )
)

## 28. Learning Curve — Validation Accuracy

In [ ]:
# Tạo figure.
plt.figure(
    figsize=(10, 6)
)

# Duyệt model.
for model_name, history in histories.items():
    # Epoch axis.
    epoch_axis = np.arange(
        1,
        len(
            history["val_acc"]
        ) + 1,
    )

    # Plot.
    plt.plot(
        epoch_axis,
        np.array(
            history["val_acc"]
        ) * 100.0,
        marker="o",
        label=model_name,
    )

# Axis.
plt.xlabel("Epoch")
plt.ylabel(
    "Validation Accuracy (%)"
)

# Title.
plt.title(
    "So sánh Validation Accuracy"
)

# Legend/grid.
plt.legend()
plt.grid(
    alpha=0.25
)

# Show.
plt.show()

## 29. Learning Curve — Validation Loss

In [ ]:
# Figure.
plt.figure(
    figsize=(10, 6)
)

# Duyệt model.
for model_name, history in histories.items():
    # Epoch axis.
    epoch_axis = np.arange(
        1,
        len(
            history["val_loss"]
        ) + 1,
    )

    # Plot.
    plt.plot(
        epoch_axis,
        history["val_loss"],
        marker="o",
        label=model_name,
    )

# Label/title.
plt.xlabel("Epoch")
plt.ylabel(
    "Validation Loss"
)
plt.title(
    "So sánh Validation Loss"
)

# Legend/grid.
plt.legend()
plt.grid(
    alpha=0.25
)

# Show.
plt.show()

## 30. So sánh Accuracy / Macro F1 / Parameters / Training Time

In [ ]:
# Tên model.
plot_models = comparison_df[
    "Model"
].tolist()

# Test accuracy.
plot_accuracy = comparison_df[
    "Test Accuracy (%)"
].to_numpy()

# Figure.
plt.figure(
    figsize=(9, 5)
)

# Bar.
plt.bar(
    plot_models,
    plot_accuracy,
)

# Label/title.
plt.ylabel(
    "Test Accuracy (%)"
)
plt.title(
    "So sánh Test Accuracy"
)
plt.xticks(
    rotation=15
)
plt.grid(
    axis="y",
    alpha=0.25,
)
plt.show()

In [ ]:
# Macro F1.
plot_f1 = comparison_df[
    "Macro F1 (%)"
].to_numpy()

# Figure.
plt.figure(
    figsize=(9, 5)
)

# Bar.
plt.bar(
    plot_models,
    plot_f1,
)

# Label/title.
plt.ylabel(
    "Macro F1 (%)"
)
plt.title(
    "So sánh Macro F1"
)
plt.xticks(
    rotation=15
)
plt.grid(
    axis="y",
    alpha=0.25,
)
plt.show()

In [ ]:
# Parameters.
plot_params = comparison_df[
    "Parameters"
].to_numpy()

# Figure.
plt.figure(
    figsize=(9, 5)
)

# Bar.
plt.bar(
    plot_models,
    plot_params,
)

# Label/title.
plt.ylabel(
    "Learnable Parameters"
)
plt.title(
    "So sánh số tham số"
)
plt.xticks(
    rotation=15
)
plt.grid(
    axis="y",
    alpha=0.25,
)
plt.show()

In [ ]:
# Training time.
plot_time = comparison_df[
    "Train Time (s)"
].to_numpy()

# Figure.
plt.figure(
    figsize=(9, 5)
)

# Bar.
plt.bar(
    plot_models,
    plot_time,
)

# Label/title.
plt.ylabel(
    "Training Time (s)"
)
plt.title(
    "So sánh thời gian huấn luyện"
)
plt.xticks(
    rotation=15
)
plt.grid(
    axis="y",
    alpha=0.25,
)
plt.show()

## 31. Confusion Matrix

In [ ]:
# Duyệt từng model.
for model_name, model in trained_models.items():
    # Prediction.
    y_true, y_pred = predict_all(
        model,
        test_loader,
        DEVICE,
    )

    # Confusion matrix.
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(
            NUM_CLASSES
        ),
    )

    # Figure.
    plt.figure(
        figsize=(8, 7)
    )

    # Image.
    plt.imshow(
        cm,
        interpolation="nearest",
    )

    # Colorbar.
    plt.colorbar()

    # Tick x.
    plt.xticks(
        np.arange(
            NUM_CLASSES
        ),
        CLASS_NAMES,
        rotation=45,
        ha="right",
    )

    # Tick y.
    plt.yticks(
        np.arange(
            NUM_CLASSES
        ),
        CLASS_NAMES,
    )

    # Axis labels.
    plt.xlabel(
        "Predicted label"
    )
    plt.ylabel(
        "True label"
    )

    # Title.
    plt.title(
        f"Confusion Matrix - {model_name}"
    )

    # Threshold.
    threshold = (
        cm.max()
        / 2.0
    )

    # Ghi số từng ô.
    for row in range(
        cm.shape[0]
    ):
        for col in range(
            cm.shape[1]
        ):
            plt.text(
                col,
                row,
                str(
                    cm[
                        row,
                        col,
                    ]
                ),
                ha="center",
                va="center",
                color=(
                    "white"
                    if cm[
                        row,
                        col,
                    ] > threshold
                    else "black"
                ),
                fontsize=9,
            )

    # Layout.
    plt.tight_layout()

    # Show.
    plt.show()

## 32. Classification Report

In [ ]:
# Duyệt model.
for model_name, model in trained_models.items():
    # Prediction.
    y_true, y_pred = predict_all(
        model,
        test_loader,
        DEVICE,
    )

    # Separator.
    print(
        "\n"
        + "=" * 80
    )

    # Tên model.
    print(
        model_name
    )

    # Report.
    print(
        classification_report(
            y_true,
            y_pred,
            labels=np.arange(
                NUM_CLASSES
            ),
            target_names=[
                str(name)
                for name in CLASS_NAMES
            ],
            digits=4,
            zero_division=0,
        )
    )

## 33. Visualization prediction một số bệnh nhân

Vì dữ liệu không phải ảnh, ta hiển thị:

- sample index
- true class
- predicted class

và xem một số feature gốc tương ứng.

In [ ]:
# Chọn model có Test Accuracy tốt nhất.
best_model_name = comparison_df.iloc[
    0
]["Model"]

# Lấy model.
best_model = trained_models[
    best_model_name
]

# Prediction toàn test.
y_true, y_pred = predict_all(
    best_model,
    test_loader,
    DEVICE,
)

# Tạo bảng preview từ raw test.
preview_count = min(
    15,
    len(
        X_test_raw
    ),
)

# Copy một số dòng raw feature.
prediction_preview = (
    X_test_raw
    .reset_index(drop=True)
    .head(
        preview_count
    )
    .copy()
)

# Thêm true label dạng chữ.
prediction_preview[
    "TRUE_DIABETES_RISK"
] = label_encoder.inverse_transform(
    y_true[
        :preview_count
    ]
)

# Thêm predicted label dạng chữ.
prediction_preview[
    "PREDICTED_DIABETES_RISK"
] = label_encoder.inverse_transform(
    y_pred[
        :preview_count
    ]
)

# Hiển thị.
display(
    prediction_preview
)

## 34. Lưu model, preprocessing pipeline, label encoder và kết quả

Đây là phần quan trọng hơn so với ảnh:

Muốn inference một bệnh nhân mới, phải dùng **đúng preprocessing pipeline đã fit trên train**.

In [ ]:
# Tạo output directory cạnh notebook.
OUTPUT_DIR = (
    Path.cwd()
    / "outputs_diabetes_cnn"
)

# Tạo folder nếu chưa có.
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Tên file model an toàn.
safe_names = {
    "CNN cơ bản": "basic_cnn_1d",
    "ResNet": "resnet_1d",
    "VGGNet": "vggnet_1d",
    "MobileNet": "mobilenet_1d",
}

# Lưu từng model.
for model_name, model in trained_models.items():
    # File path.
    model_path = (
        OUTPUT_DIR
        / f"{safe_names[model_name]}.pt"
    )

    # Chuyển state_dict về CPU.
    cpu_state_dict = {
        key: value.detach().cpu()
        for key, value
        in model.state_dict().items()
    }

    # Lưu.
    torch.save(
        cpu_state_dict,
        model_path,
    )

    # In.
    print(
        "Saved:",
        model_path,
    )

# Lưu bảng so sánh.
comparison_path = (
    OUTPUT_DIR
    / "model_comparison.csv"
)

# Save CSV.
comparison_df.to_csv(
    comparison_path,
    index=False,
)

# In.
print(
    "Saved:",
    comparison_path,
)

# Lưu preprocessing pipeline bằng pickle.
preprocessor_path = (
    OUTPUT_DIR
    / "preprocessor.pkl"
)

# Ghi object.
with open(
    preprocessor_path,
    "wb",
) as file:
    pickle.dump(
        preprocessor,
        file,
    )

# In.
print(
    "Saved:",
    preprocessor_path,
)

# Lưu label encoder.
label_encoder_path = (
    OUTPUT_DIR
    / "label_encoder.pkl"
)

# Ghi encoder.
with open(
    label_encoder_path,
    "wb",
) as file:
    pickle.dump(
        label_encoder,
        file,
    )

# In.
print(
    "Saved:",
    label_encoder_path,
)

# Lưu feature name sau preprocessing.
feature_names_path = (
    OUTPUT_DIR
    / "processed_feature_names.csv"
)

# Tạo DataFrame.
pd.DataFrame(
    {
        "feature_name": processed_feature_names
    }
).to_csv(
    feature_names_path,
    index=False,
)

# In.
print(
    "Saved:",
    feature_names_path,
)

# 35. Cách đọc kết quả

Khi viết phần **ĐÁNH GIÁ, SO SÁNH, VISUALIZATION**, không nên chỉ kết luận model nào có Accuracy lớn nhất.

Hãy phân tích đồng thời:

- **Test Accuracy**: tỷ lệ dự đoán đúng tổng thể.
- **Macro F1**: đặc biệt quan trọng khi class imbalance.
- **Validation Loss**: chất lượng hội tụ.
- **Train/Val gap**: dấu hiệu overfitting.
- **Parameters**: độ nặng của model.
- **Training Time**: chi phí thực tế.
- **Confusion Matrix**: class nào dễ nhầm.
- **Data leakage**: nếu một feature được tạo từ target, metric có thể cao giả tạo.

### Ý nghĩa bốn kiến trúc

**CNN cơ bản**
- Baseline Conv1D.
- Cho biết convolution trên chuỗi feature có học được pattern hay không.

**ResNet1D**
- Thử residual learning `F(x) + x`.
- Kiểm tra việc tăng depth có dễ tối ưu hơn không.

**VGGNet1D**
- Nhiều Conv1D kernel 3 xếp chồng.
- Có thể nhiều parameter và chậm hơn.

**MobileNet1D**
- Depthwise + Pointwise factorization.
- Kỳ vọng parameter thấp hơn.

### Lưu ý quan trọng

Thứ tự các cột tabular không có ý nghĩa không gian tự nhiên giống pixel trong ảnh. Vì vậy kết quả CNN trên Diabetes cần được diễn giải như **thử nghiệm kiến trúc**, không phải bằng chứng rằng CNN là mô hình tự nhiên nhất cho loại dữ liệu này.